# H100 네 장에서 Gemma 4와 DFlash 비교하기

같은 회의 공지를 읽고 변경된 시간을 답하는 실습입니다. 먼저 Gemma 4 31B IT만 실행합니다. 다음에는 DFlash가 여러 토큰을 미리 제안하게 하고 Gemma가 검사합니다. 답변이 공지와 맞는지, 첫 답과 전체 답을 받는 데 각각 얼마나 걸리는지 확인합니다.

**준비물**은 H100 네 장이 할당된 Linux 서버, Gemma와 이에 맞는 Z Lab DFlash 모델 폴더, 노트북과 같은 서버에서 열 수 있는 터미널입니다. 두 모델을 비교하는 동안 같은 노트북을 열어 둡니다. [4장 발표 자료](chapter-04.html#dflash)

## 1. 실행 환경 준비

이미 두 모델을 지원하는 vLLM 환경이 있다면 그 환경에서 JupyterLab을 엽니다. 새 환경을 만들 때는 아래처럼 Z Lab이 모델 카드에서 안내한 vLLM 소스를 사용합니다. 이 설치는 소스 빌드이므로 CUDA를 지원하는 Linux 빌드 환경과 CUDA Toolkit이 필요합니다. 기존 Qwen 실습 환경과는 별도로 만드세요.

```bash
uv venv --python 3.12
source .venv/bin/activate
uv pip install --torch-backend=auto "vllm @ git+https://github.com/vllm-project/vllm.git@refs/pull/41703/head" jupyterlab pandas
jupyter lab
```

노트북에서 두 번 멈춥니다. **3절**에서는 터미널에서 Gemma 서버를 켭니다. **5절**에서는 그 서버를 종료한 뒤 DFlash와 함께 다시 실행합니다. [Z Lab 설치·실행 예시](https://huggingface.co/z-lab/gemma-4-31B-it-DFlash#quick-start) · [vLLM 소스 빌드](https://docs.vllm.ai/en/stable/getting_started/installation/gpu/)

### 사진 한 픽셀로 PyTorch 확인하기

발표에 나온 컬러 사진에서 픽셀 하나의 원래 밝기가 빨강 200, 초록 100, 파랑 0이라고 가정합니다. 세 값을 각각 65%로 줄인 뒤 흑백으로 바꿉니다. 아래에서 `@`는 같은 위치의 값끼리 곱해서 모두 더합니다. 코드의 목적은 계산 결과를 확인하는 것이므로 여기서는 CPU로 실행합니다. [발표의 사진 예제](chapter-04.html#pytorch)

In [ ]:
import torch

rgb = torch.tensor([200., 100., 0.])
dark = rgb * 0.65
gray = dark @ torch.tensor([0.2126, 0.7152, 0.0722])

print("밝기를 줄인 RGB", dark.tolist())
print("흑백 밝기", round(gray.item()))

## 2. 모델 폴더와 토큰 확인하기

`TARGET`을 실제 Gemma 폴더로, `DRAFT`를 **이 Gemma용** DFlash 폴더로 바꿉니다. 코드는 가중치 파일의 이름을 색인에서 읽습니다. 아직 가중치를 GPU에 올리는 단계는 아닙니다. GPU 목록에 할당받은 H100 네 장이 보이는지도 확인합니다. [발표의 모델 파일 표](chapter-04.html#files)

In [ ]:
import json
import shlex
import subprocess
import sys
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
from time import perf_counter

import pandas as pd
import requests
from openai import OpenAI
from prometheus_client.parser import text_string_to_metric_families
from tokenizers import Tokenizer

TARGET = Path("/models/gemma-4-31B-it")
DRAFT = Path("/models/gemma-4-31B-it-DFlash")

print(subprocess.check_output(
    ["nvidia-smi", "--query-gpu=index,name,memory.total", "--format=csv"],
    text=True,
))
target_config = json.loads((TARGET / "config.json").read_text())
draft_config = json.loads((DRAFT / "config.json").read_text())
index = json.loads((TARGET / "model.safetensors.index.json").read_text())
print("Gemma 텍스트 모델 층 수", target_config["text_config"]["num_hidden_layers"])
print("가중치 파일", sorted(set(index["weight_map"].values())))
print("보조 모델", draft_config["architectures"], "블록 크기", draft_config["block_size"])

In [ ]:
tokenizer = Tokenizer.from_file(str(TARGET / "tokenizer.json"))
for sentence in ["목요일 오후 2시입니다.", "목요일 오후 4시입니다."]:
    print(sentence, tokenizer.encode(sentence, add_special_tokens=False).tokens)

두 문장에서 숫자 앞까지는 토큰이 같습니다. `▁`는 띄어쓰기입니다. 5절에서 DFlash가 **2**를 제안하고 Gemma가 **4**를 고른 경우와 연결해 봅니다. 지금 본 두 문장은 토큰을 확인하기 위한 예문입니다.

## 3. Gemma 서버를 켜고 회의 시간을 묻기

다음 셀은 서버 실행 명령을 **출력**합니다. 그 한 줄을 같은 서버의 별도 터미널에서 실행합니다. 터미널에 `Application startup complete`가 표시되면 서버를 켜 둔 채 노트북으로 돌아옵니다.

- `--tensor-parallel-size 4`는 네 GPU가 모델 하나의 계산을 나누도록 합니다.
- `--max-model-len 4096`은 한 요청에서 입력과 출력을 합해 사용할 수 있는 최대 토큰 수입니다.
- 두 설정 모두 텍스트만 처리하고, 반복되는 공지의 입력 계산을 매번 다시 합니다. 아래 명령은 1절의 Z Lab 환경용입니다. 이미 설치한 다른 버전에서 접두사 캐싱 옵션이 다르면 `vllm serve --help`에 나오는 `--disable-prefix-caching`으로 바꿉니다. 이 실습은 동시 요청 수가 바뀔 때도 같은 방식으로 계산하도록 CUDA 그래프를 끕니다(`--enforce-eager`). 이 설정에서 얻은 속도를 비교합니다. [모델 작성자의 배치 검증 논의](https://github.com/vllm-project/vllm/pull/41703) · [발표의 GPU 구성](chapter-04.html#parallel)

In [ ]:
base_url = "http://127.0.0.1:8000"
client = OpenAI(base_url=f"{base_url}/v1", api_key="local")

server = [
    sys.executable, "-m", "vllm.entrypoints.cli.main", "serve", str(TARGET),
    "--host", "127.0.0.1", "--tensor-parallel-size", "4",
    "--dtype", "bfloat16", "--max-model-len", "4096",
    "--language-model-only", "--no-enable-prefix-caching",
    "--attention-backend", "TRITON_ATTN", "--enforce-eager",
]
print(shlex.join(server))

### 답변을 받으면서 시간 재기

노트북은 답변이 오는 대로 읽습니다. 글자가 처음 도착할 때까지의 시간을 **첫 응답 시간**, 마지막 사용량 정보까지 받아 요청이 끝날 때까지를 **전체 응답 시간**으로 잽니다. 한 번에 여러 토큰이 도착할 수 있어 첫 응답 시간은 사용자가 첫 글을 볼 수 있는 시점의 측정값입니다. 서버 안에서 GPU 계산을 시작한 시각과는 다릅니다.

아래 셀의 출력에서 바뀐 요일과 시간을 확인합니다. 공지대로라면 **목요일 오후 4시**입니다.

In [ ]:
def ask(question):
    start = perf_counter()
    stream = client.chat.completions.create(
        model=str(TARGET),
        messages=[{"role": "user", "content": question}],
        temperature=0,
        max_tokens=128,
        stream=True,
        stream_options={"include_usage": True},
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    text_parts = []
    first_ms = None
    finish = None
    for chunk in stream:
        if chunk.usage:
            usage = chunk.usage
        if chunk.choices:
            choice = chunk.choices[0]
            if choice.delta.content:
                if not text_parts:
                    first_ms = (perf_counter() - start) * 1000
                text_parts.append(choice.delta.content)
            if choice.finish_reason:
                finish = choice.finish_reason

    elapsed_ms = (perf_counter() - start) * 1000
    tokens = usage.completion_tokens
    return {
        "답변": "".join(text_parts), "종료": finish,
        "입력 토큰": usage.prompt_tokens, "출력 토큰": tokens,
        "첫 응답 ms": first_ms, "전체 ms": elapsed_ms,
        "TPOT ms": (elapsed_ms - first_ms) / (tokens - 1)
        if tokens > 1 and first_ms is not None else None,
    }

notice = "목요일 회의를 오후 2시에서 4시로 변경합니다. 장소는 3층 소회의실입니다."
check_question = notice + "\n변경된 회의 시간이 언제인가요?"
baseline_answer = ask(check_question)
print(baseline_answer["답변"])

## 4. 같은 질문으로 Gemma 단독 성능 재기

공지에 대한 질문, 메일 작성, 회의록 요약 등 여덟 요청을 세 번씩 보냅니다. **동시 요청 1건**에서는 앞의 답이 끝난 뒤 다음 요청을 보냅니다. **4건**에서는 동시에 최대 네 건의 답을 기다립니다. 각 조건에서 먼저 보내는 연습 요청은 측정에서 뺍니다.

| 출력 열 | 읽는 방법 |
| --- | --- |
| 출력 토큰/초 | 24건의 총 출력 토큰 수를 전체 측정 시간(초)으로 나눕니다. |
| 첫 응답 p50 ms | 요청 전송부터 첫 글을 받을 때까지의 시간 중 가운데 값입니다. |
| TPOT p50 ms | 각 요청에서 (전체 시간 − 첫 응답 시간) ÷ (출력 토큰 수 − 1)을 구한 뒤 가운데 값을 봅니다. 출력이 한 토큰이면 계산에서 제외합니다. |
| 전체 p95 ms | 전체 응답 시간의 95번째 백분위수입니다. 느린 요청이 얼마나 걸렸는지 봅니다. |

이 요청들은 문장 끝에서 생성을 마치므로 실제 출력 길이가 서로 다를 수 있습니다. 두 설정의 토큰 수도 함께 봅니다. 24건은 실습용으로 적은 수여서 p95는 몇 건의 느린 요청에도 달라질 수 있습니다. [발표의 시간 계산 예](chapter-04.html#metrics)

In [ ]:
questions = [
    check_question,
    notice + "\n회의 장소가 어디인가요?",
    notice + "\n참석자에게 보낼 변경 안내를 두 문장으로 써 주세요.",
    "회의 메모: 시안은 민지가 금요일까지 작성하고, 준호가 월요일에 검토합니다. 담당자별 할 일을 정리해 주세요.",
    "대여 안내: 도서는 14일, 잡지는 7일간 빌릴 수 있습니다. 잡지 반납 기한을 알려 주세요.",
    "신청 안내: 워크숍 신청은 금요일 오후 6시에 마감합니다. 신청자에게 보낼 한 줄 알림을 써 주세요.",
    "스터디에는 노트북과 충전기를 가져와야 합니다. 참석자에게 보낼 준비물 안내문을 써 주세요.",
    "발표 일정: 오전에는 실습 환경 설치, 오후에는 모델 실행과 결과 토론을 합니다. 일정을 두 줄로 요약해 주세요.",
]

def measure(concurrency, counters=None):
    with ThreadPoolExecutor(max_workers=concurrency) as pool:
        list(pool.map(ask, questions[:concurrency]))
        if counters:
            before = counters()
        start = perf_counter()
        answers = list(pool.map(ask, questions * 3))
        seconds = perf_counter() - start
        if counters:
            after = counters()

    rows = pd.DataFrame(answers)
    summary = {
        "동시 요청": concurrency, "요청 수": len(rows),
        "입력 토큰 합": rows["입력 토큰"].sum(), "출력 토큰 합": rows["출력 토큰"].sum(),
        "출력 토큰/초": rows["출력 토큰"].sum() / seconds,
        "첫 응답 p50 ms": rows["첫 응답 ms"].median(),
        "TPOT p50 ms": rows["TPOT ms"].median(), "전체 p95 ms": rows["전체 ms"].quantile(.95),
    }
    if counters:
        proposed = after["제안"] - before["제안"]
        accepted = after["채택"] - before["채택"]
        summary["제안 토큰"] = proposed
        summary["채택 토큰"] = accepted
        summary["채택률 %"] = 100 * accepted / proposed if proposed else None
    return summary

In [ ]:
baseline = pd.DataFrame([measure(1), measure(4)]).set_index("동시 요청")
baseline.round(2)

## 5. DFlash를 켜고 같은 질문 다시 보내기

Gemma 서버를 실행한 **터미널에서 Ctrl+C**를 누르고 종료될 때까지 기다립니다. **노트북은 계속 열어 둡니다.** 아래 셀이 출력한 명령을 같은 터미널에서 실행합니다.

Z Lab 모델의 블록 크기는 16입니다. 이미 정해진 토큰 한 자리와 새로 제안할 자리 15개를 함께 계산합니다. 예를 들어 “목요일 오후 ”까지 제안이 맞고 숫자에서 처음 다르면 앞부분을 사용하고 숫자는 Gemma가 고른 것으로 확정합니다. 이렇게 하면 맞은 토큰 수만큼 Gemma 단독 실행에서 필요했던 계산 횟수를 줄일 수 있습니다. 후보를 만드는 데도 시간과 메모리가 들기 때문에 실제 효과는 6절에서 측정합니다. [발표의 토큰별 검사 예](chapter-04.html#verify)

In [ ]:
speculation = {
    "method": "dflash",
    "model": str(DRAFT),
    "num_speculative_tokens": draft_config["block_size"] - 1,
    "attention_backend": "FLASH_ATTN",
}
print(shlex.join(server + ["--speculative-config", json.dumps(speculation)]))

새 서버가 준비되면 아래 셀을 실행합니다. 두 답이 모두 **목요일 오후 4시**를 안내하는지 봅니다. 종료 사유 `stop`은 모델이 답을 마쳤다는 뜻이고, `length`는 지정한 길이까지 생성했다는 뜻입니다.

In [ ]:
dflash_answer = ask(check_question)
pd.DataFrame([baseline_answer, dflash_answer], index=["Gemma 단독", "DFlash"])[
    ["답변", "종료", "출력 토큰"]
]

## 6. DFlash 후보 중 실제로 쓴 토큰 세기

서버의 `/metrics` 주소에서 지금까지 제안한 토큰 수와 채택한 토큰 수를 읽습니다. 실습 시작 전후의 차이를 구하면 이번 요청에 해당하는 수를 얻습니다. 이때는 같은 서버에 다른 요청을 보내지 않습니다. 예를 들어 100개를 제안하고 70개를 썼다면 채택률은 70%입니다.

In [ ]:
def draft_counts():
    response = requests.get(f"{base_url}/metrics")
    response.raise_for_status()
    names = {
        "vllm:spec_decode_num_draft_tokens_total": "제안",
        "vllm:spec_decode_num_accepted_tokens_total": "채택",
    }
    counts = {"제안": 0, "채택": 0}
    for family in text_string_to_metric_families(response.text):
        for sample in family.samples:
            if sample.name in names:
                counts[names[sample.name]] += int(sample.value)
    return counts

In [ ]:
dflash = pd.DataFrame([measure(1, draft_counts), measure(4, draft_counts)]).set_index("동시 요청")
dflash.round(2)

**제안 토큰**이 0이면 이 측정으로는 DFlash의 동작을 확인할 수 없습니다. 서버를 시작한 터미널에서 보조 모델이 로드됐는지, `/metrics`에 두 카운터가 있는지 확인합니다. 다음 표에서 **처리량 배수**가 1보다 크면 초당 출력이 늘어난 것입니다. **TPOT 감소율**이 양수이면 첫 응답 후 토큰당 평균 시간이 줄어든 것입니다.

In [ ]:
comparison = pd.DataFrame({
    "단독 토큰/초": baseline["출력 토큰/초"],
    "DFlash 토큰/초": dflash["출력 토큰/초"],
    "처리량 배수": dflash["출력 토큰/초"] / baseline["출력 토큰/초"],
    "TPOT 감소율 %": 100 * (1 - dflash["TPOT p50 ms"] / baseline["TPOT p50 ms"]),
    "단독 전체 p95 ms": baseline["전체 p95 ms"],
    "DFlash 전체 p95 ms": dflash["전체 p95 ms"],
    "채택률 %": dflash["채택률 %"],
})
comparison.round(2)

### 결과 설명과 종료

1. **한 번에 한 건**일 때 두 설정의 TPOT와 전체 p95를 읽습니다. 답이 매우 짧으면 토큰당 시간이 줄어도 전체 시간 차이는 작을 수 있습니다.
2. **동시에 네 건**일 때 두 설정의 처리량과 전체 p95를 읽습니다. Gemma 단독으로도 여러 요청을 함께 계산하므로 한 건씩 보냈을 때와 결과가 다를 수 있습니다.
3. **채택률과 속도**를 함께 봅니다. 채택률이 높아도 후보 제안과 검증에 걸린 시간까지 포함해 빨라졌는지는 측정으로 확인해야 합니다.

발표할 때는 같은 행의 값을 읽습니다. “동시에 네 건을 보냈을 때 초당 출력은 ___토큰에서 ___토큰으로, 전체 응답 시간 p95는 ___ms에서 ___ms로 바뀌었습니다.”

서버 터미널에서 **Ctrl+C**를 눌러 GPU 사용을 마칩니다. 입력과 출력 길이를 고정해서 비교하려면 [H100 Qwen 부하 테스트](h100.ipynb)의 5~7절을 사용합니다.